# Convergencia de Monte Carlo

## Importación de librerias

In [1]:
import numpy as np
from scipy.stats import norm
import pandas as pd


## Simulación del precio subyacente

Se simula el precio terminal $S_T$ bajo la medida neutral al riesgo mediante Movimiento Browniano Geométrico:

$$
S_T = S_0 \exp\left[\left(r-\frac{\sigma^2}{2}\right)T+\sigma\sqrt{T}Z\right],
$$

donde $Z \sim N(0,1)$.

El precio de la opción se calcula como el valor presente del payoff promedio

In [2]:
def gbm(S0, r, sigma, T, n):
    Z = np.random.normal(0, 1, size=n)
    return S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)

p10 = gbm(S0=100, r=0.05, sigma=0.2, T=1, n=10)
p100 = gbm(S0=100, r=0.05, sigma=0.2, T=1, n=100)
p1000 = gbm(S0=100, r=0.05, sigma=0.2, T=1, n=1000)
p10000 = gbm(S0=100, r=0.05, sigma=0.2, T=1, n=10000)
p100000 = gbm(S0=100, r=0.05, sigma=0.2, T=1, n=100000)
p1000000 = gbm(S0=100, r=0.05, sigma=0.2, T=1, n=1000000)


def mc_price(sT, K, r, T, call=True):
    if call:
        return np.exp(-r * T) * np.mean(np.maximum(sT - K, 0))
    return np.exp(-r * T) * np.mean(np.maximum(K - sT, 0))

po10 = mc_price(p10, K=100, r=0.05, T=1, call=True)
po100 = mc_price(p100, K=100, r=0.05, T=1, call=True)
po1000 = mc_price(p1000, K=100, r=0.05, T=1, call=True)
po10000 = mc_price(p10000, K=100, r=0.05, T=1, call=True)
po100000 = mc_price(p100000, K=100, r=0.05, T=1, call=True)
po1000000 = mc_price(p1000000, K=100, r=0.05, T=1, call=True)

## Cálculo con Black-Scholes

El modelo de Black–Scholes proporciona el precio teórico de una opción europea suponiendo que el precio del activo subyacente sigue un Movimiento Browniano Geométrico, con volatilidad y tasa libre de riesgo constantes.

In [3]:
def bs(S, K, T, r, sigma, call=True):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    if call:
        price = S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)
    else:
        price = K*np.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)
    return price

price = bs(S=100, K=100, T=1, r=0.05, sigma=0.2, call=True)
bs_price = [price] * 6

## Comparación de Error
La tabla siguiente muestra el precio estimado por Monte Carlo, el precio teórico de Black–Scholes y el error absoluto para cada tamaño de muestra.

In [4]:
error = [abs(po10 - price), abs(po100 - price), abs(po1000 - price), abs(po10000 - price), abs(po100000 - price), abs(po1000000 - price)]
df = pd.DataFrame({'n': [10, 100, 1000, 10000, 100000, 1000000], 
                   'Monte Carlo Price': [po10, po100, po1000, po10000, po100000, po1000000], 
                   'Black-Scholes Price': bs_price, 
                   'Error': error})

df

,n,Monte Carlo Price,Black-Scholes Price,Error
0,10,12.583450,10.450584,2.132867
1,100,12.214426,10.450584,1.763842
2,1000,10.672100,10.450584,0.221517
3,10000,10.839739,10.450584,0.389155
4,100000,10.481316,10.450584,0.030733
5,1000000,10.445705,10.450584,0.004879


## Convergencia de Monte Carlo

Para evaluar la convergencia del método Monte Carlo, se repite la estimación del precio de la opción 50 veces para cada tamaño de muestra. En cada repetición se calcula el error absoluto entre el precio estimado por Monte Carlo y el precio teórico de Black–Scholes

Posteriormente, se obtiene el promedio y la desviación estándar de los errores observados en las 50 repeticiones.  Teóricamente, el error de un estimador Monte Carlo disminuye de forma proporcional al inverso de la raíz cuadrada del número de simulaciones:

$$
\mathbb{E}
\propto \frac{1}{\sqrt{N}}
$$

donde $N$ representa el número de simulaciones. Por ello, al aumentar $N$, se espera que tanto el error promedio como su variabilidad disminuyan.

In [5]:
def errores(n:int, simulaciones):
    error_array = np.zeros(n)
    for i in range(len(error_array)):
        p = gbm(S0=100, r=0.05, sigma=0.2, T=1, n=simulaciones)
        po = mc_price(p, K=100, r=0.05, T=1, call=True)
        error = abs(po - price)
        error_array[i] = error
    return error_array.mean(), error_array.std()
e10m, e10s = errores(n=50, simulaciones=10)
e100m, e100s = errores(n=50, simulaciones=100)
e1000m, e1000s = errores(n=50, simulaciones=1000)
e10000m, e10000s = errores(n=50, simulaciones=10000)
e100000m, e100000s = errores(n=50, simulaciones=100000)
e1000000m, e1000000s = errores(n=50, simulaciones=1000000)


df_e = pd.DataFrame({'n': [10, 100, 1000, 10000, 100000, 1000000], 
                     'Error List': [e10m, e100m, e1000m, e10000m, e100000m, e1000000m], 
                     'Error Std': [e10s, e100s, e1000s, e10000s, e100000s, e1000000s]}) 

df_e


,n,Error List,Error Std
0,10,3.373019,2.058106
1,100,1.236968,0.933185
2,1000,0.378918,0.326749
3,10000,0.103532,0.086378
4,100000,0.039794,0.026391
5,1000000,0.011359,0.008504
